# Session V: File I/O and Structured Data

Persist data using text files, JSON, and CSV, and learn how to design file-based mini databases.

## Exercise 1: Daily Journal App
**Objective:** Practice reading, writing, and appending to text files.

Build a simple daily journal application that stores entries in a plain text file (`journal.txt`). Each entry should include a timestamp and the user's text.

**Tasks:**
1. `add_entry(filename, text)` → Append a new journal entry with the current date/time to the file.
2. `read_journal(filename)` → Read and display all entries from the file. Handle the case where the file doesn't exist yet.
3. `search_journal(filename, keyword)` → Read the file and return only lines that contain the keyword (case-insensitive).
4. `count_entries(filename)` → Return the total number of entries in the journal.
5. `run_journal()` → A menu-driven interface: [A]dd entry, [R]ead all, [S]earch, [C]ount, [Q]uit.

In [ ]:
from datetime import datetime

JOURNAL_FILE = "journal.txt"

def add_entry(filename, text):
    """Append a timestamped journal entry to the file."""
    with open(filename, "a") as file:
        file.write(datetime.now().strftime("%Y-%m-%d %H:%M:%S") + " " + text + "\n")

def read_journal(filename):
    """Read and print all entries. Handle FileNotFoundError."""
    try:
        with open(filename, "r") as file:
            for line in file:
                print(line.strip())
    except FileNotFoundError:
        print("No journal entries yet.")

def search_journal(filename, keyword):
    """Return a list of lines containing the keyword (case-insensitive)."""
    try:
        with open(filename, "r") as file:
            return [line.strip() for line in file if keyword.lower() in line.lower()]
    except FileNotFoundError:
        return []

def count_entries(filename):
    """Count and return the number of journal entries."""
    try:
        with open(filename, "r") as file:
            entries = [line for line in file if line.strip()]
            return len(entries)
    except FileNotFoundError:
        return 0

def run_journal():
    """Menu-driven journal interface."""
    while True:
        print("[A]dd entry, [R]ead all, [S]earch, [C]ount, [Q]uit")
        choice = input("Choose an option: ").strip().upper()

        if choice == "A":
            text = input("Write your journal entry: ")
            add_entry(JOURNAL_FILE, text)
            print("Entry added.")
        elif choice == "R":
            read_journal(JOURNAL_FILE)
        elif choice == "S":
            keyword = input("Keyword to search: ")
            results = search_journal(JOURNAL_FILE, keyword)
            if results:
                for line in results:
                    print(line)
            else:
                print("No matching entries.")
        elif choice == "C":
            print("Total entries:", count_entries(JOURNAL_FILE))
        elif choice == "Q":
            break
        else:
            print("Please choose A, R, S, C, or Q.")


# --- QUICK TEST (non-interactive) ---
# add_entry(JOURNAL_FILE, "Started learning about file I/O today!")
# add_entry(JOURNAL_FILE, "Python makes reading files so easy.")
# read_journal(JOURNAL_FILE)
# print("Search results:", search_journal(JOURNAL_FILE, "python"))
# print("Total entries:", count_entries(JOURNAL_FILE))

# run_journal()  # Uncomment to test the interactive menu

---
## Exercise 2: To-Do List Manager (JSON Persistence)
**Objective:** Build a CRUD application that stores data in a JSON file.

Create a to-do list manager where tasks are stored as a list of dictionaries in a `tasks.json` file. Each task has a `title`, a `priority` (high/medium/low), and a `done` status (True/False).

**Tasks:**
1. `load_tasks(filename)` → Load tasks from JSON. Return an empty list if the file is missing or corrupted.
2. `save_tasks(filename, tasks)` → Save the tasks list to JSON with pretty formatting.
3. `add_task(tasks)` → Prompt for title and priority, append a new task dict (done=False).
4. `list_tasks(tasks)` → Display all tasks in a formatted table with their index, status, title, and priority.
5. `mark_done(tasks)` → Ask for a task index and set its `done` field to True.
6. `delete_task(tasks)` → Ask for a task index and remove the task from the list.
7. `run_todo()` → Menu loop: [A]dd, [L]ist, [D]one, [R]emove, [Q]uit. Auto-save after every change.

**Data structure:**
```python
[
    {"title": "Buy groceries", "priority": "high", "done": False},
    {"title": "Read chapter 5", "priority": "medium", "done": True}
]
```

In [ ]:
import json

TODO_FILE = "tasks.json"

def load_tasks(filename):
    """Load tasks from a JSON file. Return [] if missing or corrupted."""
    try:
        with open(filename, "r") as file:
            return json.load(file)
    except (FileNotFoundError, json.JSONDecodeError):
        return []

def save_tasks(filename, tasks):
    """Save tasks list to a JSON file with indent=2."""
    with open(filename, "w") as file:
        json.dump(tasks, file, indent=2)

def add_task(tasks):
    """Prompt for title and priority, append new task."""
    title = input("Task title: ").strip()
    if not title:
        print("Title cannot be empty.")
        return
    priority = input("Priority (high/medium/low): ").strip().lower()
    if priority not in ["high", "medium", "low"]:
        print("Invalid priority.")
        return
    tasks.append({"title": title, "priority": priority, "done": False})

def list_tasks(tasks):
    """Display all tasks in a formatted table."""
    if not tasks:
        print("No tasks yet.")
        return
    print(f"{'Index':<7} {'Status':<8} {'Title':<30} Priority")
    for index, task in enumerate(tasks):
        status = "Done" if task["done"] else "Pending"
        print(f"{index:<7} {status:<8} {task['title']:<30} {task['priority']}")

def mark_done(tasks):
    """Ask for a task index and mark it as done."""
    try:
        index = int(input("Task index to mark done: "))
    except ValueError:
        print("Enter a whole number.")
        return
    if 0 <= index < len(tasks):
        tasks[index]["done"] = True
    else:
        print("Invalid task index.")

def delete_task(tasks):
    """Ask for a task index and remove the task."""
    try:
        index = int(input("Task index to remove: "))
    except ValueError:
        print("Enter a whole number.")
        return
    if 0 <= index < len(tasks):
        tasks.pop(index)
    else:
        print("Invalid task index.")

def run_todo():
    """Main menu loop with auto-save."""
    tasks = load_tasks(TODO_FILE)
    while True:
        print("[A]dd, [L]ist, [D]one, [R]emove, [Q]uit")
        choice = input("Choose an option: ").strip().lower()
        if choice == "a":
            add_task(tasks)
            save_tasks(TODO_FILE, tasks)
        elif choice == "l":
            list_tasks(tasks)
        elif choice == "d":
            mark_done(tasks)
            save_tasks(TODO_FILE, tasks)
        elif choice == "r":
            delete_task(tasks)
            save_tasks(TODO_FILE, tasks)
        elif choice == "q":
            break
        else:
            print("Invalid option.")


# --- QUICK TEST (non-interactive) ---
# tasks = load_tasks(TODO_FILE)
# tasks.append({"title": "Test task", "priority": "high", "done": False})
# save_tasks(TODO_FILE, tasks)
# tasks = load_tasks(TODO_FILE)
# list_tasks(tasks)

# run_todo()  # Uncomment to test the interactive menu

---
## Exercise 3: CSV Expense Importer
**Objective:** Read, validate, and analyze data from a CSV file.

You receive a CSV file of monthly expenses. Some rows may have missing or invalid data. Write functions to import the data safely, clean it, and produce summary reports.

**First, run the cell below to generate the sample CSV file.** Then write the functions to process it.

**Tasks:**
1. `load_expenses(filename)` → Read the CSV using `csv.DictReader`. Validate each row: skip rows where `amount` is not a valid number or `category` is empty. Return a list of clean dicts (with `amount` converted to float).
2. `total_by_category(expenses)` → Return a dict mapping each category to its total spending.
3. `most_expensive(expenses)` → Return the single most expensive entry.
4. `print_report(expenses)` → Print a formatted summary: total spent, breakdown by category, and the most expensive item.

In [ ]:
import csv

# --- RUN THIS CELL FIRST to generate the sample CSV ---
sample_data = """date,description,category,amount
2024-01-05,Grocery store,Food,52.30
2024-01-06,Bus pass,Transport,40.00
2024-01-07,Coffee shop,Food,4.50
2024-01-08,Netflix,Entertainment,15.99
2024-01-09,Gas station,Transport,bad_data
2024-01-10,Restaurant,Food,38.75
2024-01-11,Gym membership,,45.00
2024-01-12,Bookstore,Education,22.50
2024-01-13,Electricity bill,Utilities,89.00
2024-01-14,Movie tickets,Entertainment,24.00
2024-01-15,Taxi,Transport,18.50
2024-01-16,Online course,Education,49.99
2024-01-17,Supermarket,Food,67.20
2024-01-18,Water bill,Utilities,not_a_number
2024-01-19,Concert,Entertainment,75.00
2024-01-20,Pharmacy,Health,12.40"""

with open("expenses.csv", "w", newline="") as f:
    f.write(sample_data)

print("✅ expenses.csv created successfully.")

In [ ]:
import csv

def load_expenses(filename):
    """Load and validate expenses from CSV. Skip invalid rows."""
    expenses_list = []
    try:
        with open(filename, "r", newline="") as file:
            reader = csv.DictReader(file)
            for row in reader:
                try:
                    amount = float(row.get("amount", ""))
                except (ValueError, TypeError):
                    continue
                category = (row.get("category") or "").strip()
                if not category:
                    continue
                expenses_list.append({
                    "date": row.get("date", ""),
                    "description": row.get("description", ""),
                    "category": category,
                    "amount": amount,
                })
    except FileNotFoundError:
        print("Expense file not found.")
    return expenses_list

def total_by_category(expenses):
    """Return {category: total_amount} dict."""
    totals = {}
    for expense in expenses:
        category = expense["category"]
        totals[category] = totals.get(category, 0) + expense["amount"]
    return totals

def most_expensive(expenses):
    """Return the expense dict with the highest amount."""
    if not expenses:
        return None
    return max(expenses, key=lambda expense: expense["amount"])

def print_report(expenses):
    """Print a formatted expense summary report."""
    total = sum(expense["amount"] for expense in expenses)
    print(f"Total spent: ${total:.2f}")
    print("Spending by category:")
    for category, amount in total_by_category(expenses).items():
        print(f"  {category}: ${amount:.2f}")
    highest = most_expensive(expenses)
    if highest is None:
        print("No expenses to report.")
    else:
        print(f"Most expensive: {highest['description']} (${highest['amount']:.2f})")


# --- TEST ---
expenses = load_expenses("expenses.csv")
print_report(expenses)

---
## Exercise 4: File Content Validator
**Objective:** Write robust validation functions for both JSON and CSV files.

You will build a validation toolkit that checks whether files are well-formed and contain the expected structure *before* your application tries to use the data.

**First, run the cell below to generate sample test files** (some valid, some intentionally broken). Then write the validators.

**Tasks:**
1. `validate_json(filename, required_keys)` → Check that the file (a) exists, (b) contains valid JSON, (c) is a list of dicts, and (d) each dict contains all `required_keys`. Return a report dict: `{"valid": int, "invalid": int, "errors": [str]}`.
2. `validate_csv(filename, expected_columns)` → Check that the file (a) exists, (b) has the correct number of columns in the header, and (c) each data row has the correct number of fields. Return a similar report dict.
3. `print_validation_report(report, filename)` → Pretty-print the validation results.

In [ ]:
import json

# --- RUN THIS CELL FIRST to generate test files ---

# Good JSON file
good_json = [
    {"name": "Alice", "age": 22, "email": "alice@test.com"},
    {"name": "Bob", "age": 25, "email": "bob@test.com"},
    {"name": "Charlie", "age": 30, "email": "charlie@test.com"},
]
with open("good_data.json", "w") as f:
    json.dump(good_json, f, indent=2)

# Bad JSON file (missing keys, wrong types)
bad_json = [
    {"name": "Alice", "age": 22, "email": "alice@test.com"},
    {"name": "Bob", "age": 25},
    "this is not a dict",
    {"name": "Diana", "age": "not_a_number", "email": "diana@test.com"},
]
with open("bad_data.json", "w") as f:
    json.dump(bad_json, f, indent=2)

# Corrupted JSON file
with open("corrupted.json", "w") as f:
    f.write('{"name": "broken", missing_quotes: true}')

# Good CSV file
with open("good_data.csv", "w", newline="") as f:
    f.write("name,age,email\n")
    f.write("Alice,22,alice@test.com\n")
    f.write("Bob,25,bob@test.com\n")

# Bad CSV file (inconsistent columns)
with open("bad_data.csv", "w", newline="") as f:
    f.write("name,age,email\n")
    f.write("Alice,22,alice@test.com\n")
    f.write("Bob,25\n")
    f.write("Charlie,30,charlie@test.com,extra_field\n")
    f.write("Diana,28,diana@test.com\n")

print("✅ All test files created successfully.")

In [ ]:
import json
import csv

def validate_json(filename, required_keys):
    """Validate a JSON file's structure and required keys.
    Return {"valid": int, "invalid": int, "errors": [str]}."""
    report = {"valid": 0, "invalid": 0, "errors": []}
    try:
        with open(filename, "r") as file:
            data = json.load(file)
    except (FileNotFoundError, json.JSONDecodeError) as error:
        report["invalid"] = 1
        report["errors"].append(str(error))
        return report
    if not isinstance(data, list):
        report["invalid"] = 1
        report["errors"].append("JSON root must be a list.")
        return report
    for index, record in enumerate(data):
        if not isinstance(record, dict):
            report["invalid"] += 1
            report["errors"].append(f"Item {index}: expected a dictionary.")
            continue
        missing = [key for key in required_keys if key not in record]
        if missing:
            report["invalid"] += 1
            report["errors"].append(f"Item {index}: missing keys {missing}.")
        else:
            report["valid"] += 1
    return report

def validate_csv(filename, expected_columns):
    """Validate a CSV file's structure and column count.
    Return {"valid": int, "invalid": int, "errors": [str]}."""
    report = {"valid": 0, "invalid": 0, "errors": []}
    try:
        with open(filename, "r", newline="") as file:
            reader = csv.reader(file)
            header = next(reader, None)
            if header is None or len(header) != expected_columns:
                report["invalid"] += 1
                report["errors"].append("Header has an incorrect number of columns.")
            for row_number, row in enumerate(reader, start=2):
                if len(row) == expected_columns:
                    report["valid"] += 1
                else:
                    report["invalid"] += 1
                    report["errors"].append(f"Row {row_number}: incorrect number of fields.")
    except (FileNotFoundError, csv.Error) as error:
        report["invalid"] += 1
        report["errors"].append(str(error))
    return report

def print_validation_report(report, filename):
    """Pretty-print the validation results."""
    print(f"Validation report: {filename}")
    print(f"Valid: {report['valid']}, invalid: {report['invalid']}")
    for error in report["errors"]:
        print(f"  {error}")


# --- TEST ---
print("=" * 50)
report = validate_json("good_data.json", ["name", "age", "email"])
print_validation_report(report, "good_data.json")

print("=" * 50)
report = validate_json("bad_data.json", ["name", "age", "email"])
print_validation_report(report, "bad_data.json")

print("=" * 50)
report = validate_json("corrupted.json", ["name", "age", "email"])
print_validation_report(report, "corrupted.json")

print("=" * 50)
report = validate_csv("good_data.csv", 3)
print_validation_report(report, "good_data.csv")

print("=" * 50)
report = validate_csv("bad_data.csv", 3)
print_validation_report(report, "bad_data.csv")

---
## Exercise 5: Format Converter & Exporter
**Objective:** Convert data between JSON and CSV formats in both directions.

Build a bidirectional converter that can transform data files between JSON and CSV. This is an extremely common real-world task — APIs return JSON, but spreadsheets need CSV, and vice versa.

**Tasks:**
1. `json_to_csv(json_filename, csv_filename)` → Read a JSON file (list of dicts) and write it as a CSV file. Use the keys of the first dict as column headers.
2. `csv_to_json(csv_filename, json_filename)` → Read a CSV file and write it as a JSON file (list of dicts).
3. `export_filtered_csv(json_filename, csv_filename, filter_key, filter_value)` → Read a JSON file, keep only records where `record[filter_key] == filter_value`, and export the filtered data to CSV.
4. Handle edge cases: empty files, missing files, and mismatched structures.

In [ ]:
import json
import csv

# --- Setup: create a test JSON file ---
students = [
    {"name": "Alice", "age": 22, "city": "Madrid", "grade": "A"},
    {"name": "Bob", "age": 25, "city": "London", "grade": "B"},
    {"name": "Charlie", "age": 30, "city": "Paris", "grade": "A"},
    {"name": "Diana", "age": 28, "city": "Madrid", "grade": "C"},
    {"name": "Eve", "age": 23, "city": "London", "grade": "A"},
]

with open("students.json", "w") as f:
    json.dump(students, f, indent=2)

print("✅ students.json created.")

In [ ]:
import json
import csv

def json_to_csv(json_filename, csv_filename):
    """Convert a JSON file (list of dicts) to a CSV file."""
    try:
        with open(json_filename, "r") as file:
            data = json.load(file)
        if not isinstance(data, list) or any(not isinstance(record, dict) for record in data):
            print("Expected a JSON list of dictionaries.")
            return
        if not data:
            with open(csv_filename, "w", newline="") as file:
                file.write("")
            print("No records to export.")
            return
        fieldnames = list(data[0].keys())
        if not fieldnames or any(set(record) != set(fieldnames) for record in data):
            print("Records must have the same non-empty set of keys.")
            return
        with open(csv_filename, "w", newline="") as file:
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(data)
    except (FileNotFoundError, json.JSONDecodeError, csv.Error) as error:
        print(f"Conversion failed: {error}")

def csv_to_json(csv_filename, json_filename):
    """Convert a CSV file to a JSON file (list of dicts)."""
    try:
        with open(csv_filename, "r", newline="") as file:
            reader = csv.DictReader(file)
            fieldnames = reader.fieldnames
            if fieldnames is not None and (
                any(not name.strip() for name in fieldnames)
                or len(set(fieldnames)) != len(fieldnames)
            ):
                print("CSV headers must be non-empty and unique.")
                return
            records = []
            for row in reader:
                if None in row or any(value is None for value in row.values()):
                    print("CSV rows do not match the header.")
                    return
                records.append(row)
        with open(json_filename, "w") as file:
            json.dump(records, file, indent=2)
    except (FileNotFoundError, csv.Error) as error:
        print(f"Conversion failed: {error}")

def export_filtered_csv(json_filename, csv_filename, filter_key, filter_value):
    """Export only records where record[filter_key] == filter_value to CSV."""
    try:
        with open(json_filename, "r") as file:
            data = json.load(file)
        if not isinstance(data, list) or any(not isinstance(record, dict) for record in data):
            print("Expected a JSON list of dictionaries.")
            return
        if not data:
            with open(csv_filename, "w", newline="") as file:
                file.write("")
            print("No records to export.")
            return
        fieldnames = list(data[0].keys())
        if not fieldnames or any(set(record) != set(fieldnames) for record in data):
            print("Records must have the same non-empty set of keys.")
            return
        if filter_key not in fieldnames:
            print("Filter key is missing from the records.")
            return
        filtered = [record for record in data if record[filter_key] == filter_value]
        with open(csv_filename, "w", newline="") as file:
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(filtered)
        if not filtered:
            print("No matching records.")
    except (FileNotFoundError, json.JSONDecodeError, csv.Error) as error:
        print(f"Export failed: {error}")


# --- TEST ---
# JSON → CSV
json_to_csv("students.json", "students_export.csv")

# CSV → JSON (round-trip test)
csv_to_json("students_export.csv", "students_roundtrip.json")

# Filtered export: only students from Madrid
export_filtered_csv("students.json", "madrid_students.csv", "city", "Madrid")

# Verify round-trip
with open("students_roundtrip.json", "r") as f:
    roundtrip = json.load(f)
print(f"\nRound-trip data ({len(roundtrip)} records):")
for r in roundtrip:
    print(f"  {r}")

---
## Exercise 6: Student Records Database (Capstone)
**Objective:** Build a complete file-based CRUD application combining JSON persistence, CSV import/export, and data validation.

Create a student records management system that stores data in a JSON file and supports importing/exporting CSV. This exercise combines **every concept** from this session.

**Required features:**
1. **[A]dd Student** — Prompt for name, age, email, and grade. Validate all inputs (age must be a positive integer, email must contain `@`, grade must be A/B/C/D/F). Store in JSON.
2. **[L]ist Students** — Display all students in a formatted table.
3. **[S]earch** — Find a student by name (case-insensitive partial match).
4. **[U]pdate** — Modify a student's grade or email by name.
5. **[D]elete** — Remove a student by name.
6. **[I]mport CSV** — Load students from a CSV file, validating each row before adding. Skip invalid rows and report what was skipped.
7. **[E]xport CSV** — Export all current students to a CSV file.
8. **[Q]uit** — Exit the program.

**Data structure (JSON):**
```python
[
    {"name": "Alice", "age": 22, "email": "alice@uni.edu", "grade": "A"},
    {"name": "Bob", "age": 25, "email": "bob@uni.edu", "grade": "B"}
]
```

In [ ]:
import json
import csv

DB_FILE = "students_db.json"

# --- PERSISTENCE ---

def load_db(filename):
    """Load student records from JSON. Return [] if missing/corrupted."""
    try:
        with open(filename, "r") as file:
            students = json.load(file)
        if not isinstance(students, list):
            return []
        valid_students = []
        for student in students:
            if not isinstance(student, dict):
                continue
            if any(key not in student for key in ["name", "age", "email", "grade"]):
                continue
            valid, error = validate_student(
                student["name"], student["age"], student["email"], student["grade"]
            )
            if valid:
                valid_students.append(student)
        return valid_students
    except (FileNotFoundError, json.JSONDecodeError):
        return []

def save_db(filename, students):
    """Save student records to JSON."""
    with open(filename, "w") as file:
        json.dump(students, file, indent=2)

# --- VALIDATION ---

def validate_student(name, age, email, grade):
    """Validate student fields. Return (is_valid, error_message)."""
    if not isinstance(name, str) or not name.strip():
        return False, "Name cannot be empty."
    if not isinstance(age, int) or isinstance(age, bool) or age <= 0:
        return False, "Age must be a positive integer."
    if not isinstance(email, str) or "@" not in email:
        return False, "Email must contain @."
    if grade not in ["A", "B", "C", "D", "F"]:
        return False, "Grade must be A, B, C, D, or F."
    return True, ""

# --- CRUD OPERATIONS ---

def add_student(students):
    """Prompt for student info, validate, and add to list."""
    name = input("Name: ").strip()
    try:
        age = int(input("Age: "))
    except ValueError:
        print("Age must be a positive integer.")
        return
    email = input("Email: ").strip()
    grade = input("Grade (A/B/C/D/F): ").strip().upper()
    valid, error = validate_student(name, age, email, grade)
    if not valid:
        print(error)
        return
    students.append({"name": name, "age": age, "email": email, "grade": grade})
    print("Student added.")

def list_students(students):
    """Display all students in a formatted table."""
    if not students:
        print("No students found.")
        return
    print(f"{'Name':<20} {'Age':<5} {'Email':<30} Grade")
    for student in students:
        print(f"{student['name']:<20} {student['age']:<5} {student['email']:<30} {student['grade']}")

def search_student(students):
    """Search by name (case-insensitive partial match)."""
    term = input("Name to search: ").strip().lower()
    matches = [student for student in students if term in student["name"].lower()]
    list_students(matches)
    return matches

def update_student(students):
    """Update a student's grade or email by name."""
    name = input("Name of student to update: ").strip().lower()
    for student in students:
        if student["name"].lower() == name:
            field = input("Update grade or email? ").strip().lower()
            if field not in ["grade", "email"]:
                print("Choose grade or email.")
                return
            value = input("New value: ").strip()
            grade = value.upper() if field == "grade" else student["grade"]
            email = value if field == "email" else student["email"]
            valid, error = validate_student(student["name"], student["age"], email, grade)
            if not valid:
                print(error)
                return
            student[field] = grade if field == "grade" else email
            print("Student updated.")
            return
    print("Student not found.")

def delete_student(students):
    """Remove a student by name."""
    name = input("Name of student to delete: ").strip().lower()
    for index, student in enumerate(students):
        if student["name"].lower() == name:
            confirm = input(f"Delete {student['name']}? (yes/no): ").strip().lower()
            if confirm == "yes":
                students.pop(index)
                print("Student deleted.")
            else:
                print("Deletion cancelled.")
            return
    print("Student not found.")

# --- IMPORT / EXPORT ---

def import_csv(students, csv_filename):
    """Import students from CSV, validating each row."""
    imported = 0
    try:
        with open(csv_filename, "r", newline="") as file:
            reader = csv.DictReader(file)
            required = ["name", "age", "email", "grade"]
            if reader.fieldnames is None or any(key not in reader.fieldnames for key in required):
                print("CSV must include name, age, email, and grade columns.")
                return
            for row_number, row in enumerate(reader, start=2):
                if None in row or any(row.get(key) is None for key in required):
                    print(f"Skipped row {row_number}: incorrect number of fields.")
                    continue
                try:
                    age = int(row["age"])
                except ValueError:
                    print(f"Skipped row {row_number}: age must be a positive integer.")
                    continue
                name = row["name"].strip()
                email = row["email"].strip()
                grade = row["grade"].strip().upper()
                valid, error = validate_student(name, age, email, grade)
                if not valid:
                    print(f"Skipped row {row_number}: {error}")
                    continue
                students.append({"name": name, "age": age, "email": email, "grade": grade})
                imported += 1
        print(f"Imported {imported} student(s).")
    except (FileNotFoundError, csv.Error) as error:
        print(f"Import failed: {error}")

def export_csv(students, csv_filename):
    """Export all students to a CSV file."""
    with open(csv_filename, "w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=["name", "age", "email", "grade"])
        writer.writeheader()
        writer.writerows(students)
    print(f"Exported {len(students)} student(s).")

# --- MAIN MENU ---

def run_student_db():
    """Main menu loop."""
    students = load_db(DB_FILE)
    print(f"Loaded {len(students)} student(s) from database.")

    while True:
        print("\n===== STUDENT RECORDS DATABASE =====")
        print("[A]dd  [L]ist  [S]earch  [U]pdate  [D]elete")
        print("[I]mport CSV  [E]xport CSV  [Q]uit")
        choice = input("> ").strip().lower()

        if choice == "a":
            add_student(students)
            save_db(DB_FILE, students)
        elif choice == "l":
            list_students(students)
        elif choice == "s":
            search_student(students)
        elif choice == "u":
            update_student(students)
            save_db(DB_FILE, students)
        elif choice == "d":
            delete_student(students)
            save_db(DB_FILE, students)
        elif choice == "i":
            csv_file = input("CSV filename to import: ").strip()
            import_csv(students, csv_file)
            save_db(DB_FILE, students)
        elif choice == "e":
            csv_file = input("CSV filename to export to: ").strip()
            export_csv(students, csv_file)
        elif choice == "q":
            save_db(DB_FILE, students)
            print("Database saved. Goodbye!")
            break
        else:
            print("Invalid option. Please try again.")

# run_student_db()  # Uncomment to test

In [ ]:
# --- Generate a sample CSV for testing the Import feature ---

sample_import = """name,age,email,grade
Frank,21,frank@uni.edu,B
Grace,23,grace@uni.edu,A
Hank,not_a_number,hank@uni.edu,C
Ivy,20,ivy_no_at_sign,A
Jack,22,jack@uni.edu,Z
Karen,24,karen@uni.edu,B"""

with open("import_students.csv", "w", newline="") as f:
    f.write(sample_import)

print("import_students.csv created (3 valid, 3 invalid rows).")